# 02 — Text preprocessing for a BiLSTM

Prepare the two-class Sentiment140 training file without changing its original text. This notebook uses light cleaning, removes conflicting and repeated normalized texts, makes stratified train/validation/test splits, and builds token IDs from training text only. No model is trained here.

In [1]:
from collections import Counter
from html import unescape
from pathlib import Path
import json
import re

import numpy as np
import pandas as pd

RANDOM_STATE = 42
DATA_PATH = Path("data/training.1600000.processed.noemoticon.csv")
COLUMNS = ["target", "tweet_id", "date", "query", "user", "text"]

## 1. Load and validate labels

The original `text` stays available for comparison. This training file is expected to contain only negative (`0`) and positive (`4`) targets.

In [2]:
if not DATA_PATH.exists():
    raise FileNotFoundError(DATA_PATH.resolve())

df = pd.read_csv(DATA_PATH, names=COLUMNS, encoding="latin-1")
unexpected = set(df["target"].dropna().unique()) - {0, 4}
if unexpected or df["target"].isna().any() or df["text"].isna().any():
    raise ValueError(f"Unexpected targets or missing values; unexpected targets: {unexpected}")
df["sentiment"] = df["target"].map({0: "negative", 4: "positive"})
print(f"Loaded {len(df):,} rows")
display(df["sentiment"].value_counts())

Loaded 1,600,000 rows


sentiment
negative    800000
positive    800000
Name: count, dtype: int64

## 2. Noice removal and cleaning

Decode HTML entities, lowercase, replace URLs and mentions with marker tokens, and normalize whitespace. Keep contractions, hashtags, negations, punctuation, and emoticons. We can test further cleaning later instead of discarding those signals now.

In [3]:
URL_RE = re.compile(r"https?://\S+|www\.\S+", re.IGNORECASE)
MENTION_RE = re.compile(r"(?<!\w)@\w+")
SPACE_RE = re.compile(r"\s+")

def clean_tweet(text):
    text = unescape(text).lower()
    text = URL_RE.sub(" <url> ", text)
    text = MENTION_RE.sub(" <user> ", text)
    return SPACE_RE.sub(" ", text).strip()

df["clean_text"] = df["text"].map(clean_tweet)
print(f"Empty cleaned texts: {df['clean_text'].eq('').sum():,}")
display(df.loc[:, ["text", "clean_text"]].sample(10, random_state=RANDOM_STATE))

Empty cleaned texts: 0


,text,clean_text
541200,@chrishasboobs AHHH I HOPE YOUR OK!!!,<user> ahhh i hope your ok!!!
750,"@misstoriblack cool , i have no tweet apps fo...","<user> cool , i have no tweet apps for my razr 2"
766711,@TiannaChaos i know just family drama. its la...,<user> i know just family drama. its lame.hey ...
285055,School email won't open and I have geography ...,school email won't open and i have geography s...
705995,upper airways problem,upper airways problem
379611,Going to miss Pastor's sermon on Faith...,going to miss pastor's sermon on faith...
1189018,on lunch....dj should come eat with me,on lunch....dj should come eat with me
667030,@piginthepoke oh why are you feeling like that?,<user> oh why are you feeling like that?
93541,gahh noo!peyton needs to live!this is horrible,gahh noo!peyton needs to live!this is horrible
1097326,@mrstessyman thank you glad you like it! There...,<user> thank you glad you like it! there is a ...


## 3. Resolve repeated normalized texts

If the same cleaned text has opposing labels, remove every copy because the text alone cannot identify one label. For other repeated cleaned texts, keep one copy. This is a modelling policy; the exploration notebook keeps the raw data intact.

In [4]:
original_rows = len(df)
df = df.loc[df["clean_text"].ne("")].copy()
empty_removed = original_rows - len(df)
label_count = df.groupby("clean_text")["sentiment"].nunique()
conflicting_texts = label_count.index[label_count.gt(1)]
non_conflicting = df.loc[~df["clean_text"].isin(conflicting_texts)]
conflict_rows_removed = len(df) - len(non_conflicting)
df = non_conflicting.drop_duplicates(subset="clean_text").copy()
same_label_copies_removed = len(non_conflicting) - len(df)
print(f"Empty cleaned texts removed: {empty_removed:,}")
print(f"Conflicting cleaned texts: {len(conflicting_texts):,}")
print(f"Rows with conflicting labels removed: {conflict_rows_removed:,}")
print(f"Extra same-label copies removed: {same_label_copies_removed:,}")
print(f"Rows retained: {len(df):,}")

Empty cleaned texts removed: 0
Conflicting cleaned texts: 3,327
Rows with conflicting labels removed: 23,757
Extra same-label copies removed: 34,952
Rows retained: 1,541,291


## 4. Stratified train/validation/test split

Use an 80/10/10 split with a fixed random seed. Since normalized duplicates were resolved first, the same cleaned tweet cannot cross split boundaries. The test set stays untouched while choosing preprocessing and model settings.

In [5]:
rng = np.random.default_rng(RANDOM_STATE)
split_indices = {"train": [], "validation": [], "test": []}
for sentiment, group in df.groupby("sentiment"):
    indices = rng.permutation(group.index.to_numpy())
    n_train = int(len(indices) * 0.80)
    n_validation = int(len(indices) * 0.10)
    split_indices["train"].extend(indices[:n_train])
    split_indices["validation"].extend(indices[n_train:n_train + n_validation])
    split_indices["test"].extend(indices[n_train + n_validation:])

splits = {name: df.loc[indices].reset_index(drop=True) for name, indices in split_indices.items()}
for name, part in splits.items():
    print(f"{name}: {len(part):,} rows")
    display(part["sentiment"].value_counts(normalize=True).mul(100).round(2))
assert sum(map(len, splits.values())) == len(df)
assert sum(part["clean_text"].nunique() for part in splits.values()) == df["clean_text"].nunique()

train: 1,233,032 rows


sentiment
negative    50.24
positive    49.76
Name: proportion, dtype: float64

validation: 154,128 rows


sentiment
negative    50.24
positive    49.76
Name: proportion, dtype: float64

test: 154,131 rows


sentiment
negative    50.24
positive    49.76
Name: proportion, dtype: float64

## 5. Tokenize and build a training-only vocabulary

Keep common contractions and hashtags as single tokens. Preserve simple emoticons and punctuation as tokens. ID `0` is padding and ID `1` is for unseen tokens. A future embedding layer turns these IDs into learned vectors; stemming and lemmatization are omitted initially.

In [6]:
TOKEN_RE = re.compile(r"<url>|<user>|[:;=8][-^]?[)(dDpP/\\]|#\w+|\w+(?:['’]\w+)*|[^\w\s]", re.IGNORECASE)

def tokenize(text):
    return TOKEN_RE.findall(text)

MAX_VOCAB_SIZE = 40_000
token_counts = Counter(token for text in splits["train"]["clean_text"] for token in tokenize(text))
vocab = {"<pad>": 0, "<unk>": 1}
for token, _ in token_counts.most_common(MAX_VOCAB_SIZE - len(vocab)):
    if token not in vocab:
        vocab[token] = len(vocab)
print(f"Vocabulary size: {len(vocab):,}")
display(pd.DataFrame({"text": splits['train']['clean_text'].head(5), "tokens": splits['train']['clean_text'].head(5).map(tokenize)}))

Vocabulary size: 40,000


,text,tokens
0,my two best friends have boyfriends. when am i...,"[my, two, best, friends, have, boyfriends, ., ..."
1,"yep, need to change to 'kel' now.","[yep, ,, need, to, change, to, ', kel, ', now, .]"
2,didn't get to enjoy lunch w/ <user> duet to th...,"[didn't, get, to, enjoy, lunch, w, /, <user>, ..."
3,asus pulled all their andriois !eeepc models f...,"[asus, pulled, all, their, andriois, !, eeepc,..."
4,alright... nick.... miley.... niley.... d: :[ ...,"[alright, ., ., ., nick, ., ., ., ., miley, .,..."


## 6. Encode and pad sequences

Choose a sequence length from the 95th percentile of training token counts. Long tweets are truncated on the right; shorter ones are padded on the right with zero. Review this length before fitting the BiLSTM.

In [7]:
train_lengths = splits["train"]["clean_text"].map(lambda text: len(tokenize(text)))
MAX_LENGTH = max(1, int(np.ceil(train_lengths.quantile(0.95))))
print(f"Sequence length (training 95th percentile): {MAX_LENGTH}")
print(f"Training tweets truncated: {train_lengths.gt(MAX_LENGTH).mean() * 100:.2f}%")

def encode_texts(texts):
    encoded = np.zeros((len(texts), MAX_LENGTH), dtype=np.int32)
    for row, text in enumerate(texts):
        ids = [vocab.get(token, 1) for token in tokenize(text)[:MAX_LENGTH]]
        encoded[row, :len(ids)] = ids
    return encoded

X = {name: encode_texts(part["clean_text"]) for name, part in splits.items()}
y = {name: part["target"].eq(4).to_numpy(dtype=np.int8) for name, part in splits.items()}
for name in splits:
    print(f"{name}: X {X[name].shape}, y {y[name].shape}")
assert all(len(X[name]) == len(y[name]) for name in splits)

Sequence length (training 95th percentile): 31
Training tweets truncated: 3.96%
train: X (1233032, 31), y (1233032,)
validation: X (154128, 31), y (154128,)
test: X (154131, 31), y (154131,)


## 7. Save the prepared data

Run this cell after reviewing the preprocessing results. Notebook 03 loads these files to train the BiLSTM without repeating the full cleaning and tokenization process.

In [8]:
PREPARED_DIR = Path("data/prepared")
PREPARED_DIR.mkdir(parents=True, exist_ok=True)
for name in splits:
    np.save(PREPARED_DIR / f"X_{name}.npy", X[name])
    np.save(PREPARED_DIR / f"y_{name}.npy", y[name])
with (PREPARED_DIR / "vocab.json").open("w", encoding="utf-8") as file:
    json.dump(vocab, file, ensure_ascii=False)
with (PREPARED_DIR / "metadata.json").open("w", encoding="utf-8") as file:
    json.dump({"max_length": MAX_LENGTH, "vocab_size": len(vocab), "random_state": RANDOM_STATE}, file, indent=2)
print(f"Saved arrays and vocabulary to {PREPARED_DIR.resolve()}")

Saved arrays and vocabulary to /Users/tajeswarramesh/repositories/scm-sentiment-analysis/data/prepared
